# A Shift-Reduce Parser for Arithmetic Expressions

In this notebook we implement a generic *shift reduce parser*.  The parse table that we use 
implements the following grammar for arithmetic expressions:
$$
  \begin{eqnarray*}
  \mathrm{expr}        & \rightarrow & \mathrm{expr}\;\;\texttt{'+'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{expr}\;\;\texttt{'-'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{product}                                    \\[0.2cm]
  \mathrm{product}     & \rightarrow & \mathrm{product}\;\;\texttt{'*'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{product}\;\;\texttt{'/'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{factor}                                     \\[0.2cm]
  \mathrm{factor}      & \rightarrow & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}     \\
                       & \mid        & \texttt{NUMBER} 
  \end{eqnarray*}
$$

## Implementing a Scanner

In order to parse, we need a scanner.  We will use a scanner that is similar to the one that we have already used for our *top down parser* discussed earlier in Chapter 6.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "ParseTable.fsx"
open ParseTable
open SLR
open GrammarSyntax
open System
open System.Text.RegularExpressions

type InputToken = { Kind: string; Lexeme: string }
let tokenize (source: string) =
    let scanner = Regex(@"\G(?:\s+|[1-9][0-9]*|0|[-+*/()])")
    let rec loop position result =
        if position = source.Length then List.rev result
        else
            let matched = scanner.Match(source, position)
            if not matched.Success then failwithf "Illegal character at column %d" (position + 1)
            let next = position + matched.Length
            if String.IsNullOrWhiteSpace matched.Value then loop next result
            else
                let kind = if Char.IsDigit matched.Value[0] then "NUMBER" else matched.Value
                loop next ({ Kind = kind; Lexeme = matched.Value } :: result)
    loop 0 []

The .NET regular-expression scanner returns token kinds and original lexemes. The parser uses kinds to select actions and preserves lexemes for diagnostics and trees.

In [ ]:
printfn "%A" (tokenize "11 + 22 * (33 - 45) / 007")

Assume a grammar $G = \langle V, T, R, S \rangle$ is given.  A  *shift-reduce parser*
is defined as a 4-Tuple
$$P = \langle Q, q_0, \texttt{action}, \texttt{goto} \rangle$$
where
- $Q$ is the set of *states* of the shift-reduce parser.  

  For the purpose of the shift-reduce-parser, states are purely abstract. 
- $q_0 \in Q$ is the *start state*.
- $\texttt{action}$ is a function taking two arguments. The first argument is a state $q \in Q$
  and the second argument is a token $t \in T$.  The result of this function is an element from the set
  $$\texttt{Action} :=
       \bigl\{ \langle\texttt{shift}, q\rangle  \mid q \in Q \bigr\}               \cup 
       \bigl\{ \langle\texttt{reduce}, r\rangle \mid r \in R \bigr\} \cup 
       \bigl\{ \texttt{accept} \bigr\}                        \cup
       \bigl\{ \texttt{error}  \bigr\}.                         
  $$
  Here `shift`, `reduce`, `accept`, and `error` are strings that serve to
  distinguish the different kinds of results returned by the function 
  `action`:
  $$\texttt{action}: Q \times T \rightarrow \texttt{Action}.$$
- `goto` is a function that takes a state $q \in Q$ and a syntactical variable
  $v \in V$ and computes a new state:
  $$\texttt{goto}: Q \times V \rightarrow Q.$$

The `ParseTable` record maintains two immutable maps:
- `mActionTable` encodes the function $\texttt{action}: Q \times T \rightarrow \texttt{Action}$.
- `mGotoTable` encodes the function $\texttt{goto}: Q \times V \rightarrow Q$.

The constructor takes these tables as arguments and stores them in the member variables `mActionTable` and `mGotoTable`.

`parseBy` exposes exactly two semantic operations: turn a shifted token into a value, and combine a reduced production’s child values. The recognizer uses `unit`; the next notebook uses trees.

`parse` returns `true` for an accepted token list and `false` for syntax errors. The reusable `parseBy` also returns a useful error message and rejects table conflicts.

In [ ]:
// The state stack has one more entry than the semantic-value stack.
let parseBy table leaf reduce tokens =
    let rec step states values remaining =
        match states, remaining with
        | state :: _, token :: tail ->
            match Map.tryFind (state, token.Kind) table.Actions with
            | Some candidates when Set.count candidates = 1 ->
                match Set.minElement candidates with
                | Shift target -> step (target :: states) (leaf token :: values) tail
                | Reduce rule ->
                    let count = rule.Body.Length
                    if count >= states.Length || count > values.Length then Error "Invalid reduction stack"
                    else
                        let children = values |> List.take count |> List.rev
                        let restStates, restValues = List.skip count states, List.skip count values
                        match Map.tryFind (restStates.Head, rule.Head) table.Gotos with
                        | Some target -> step (target :: restStates) (reduce rule children :: restValues) remaining
                        | None -> Error(sprintf "Missing goto for %s" rule.Head)
                | Accept ->
                    match values, remaining with
                    | [value], [last] when last.Kind = "$" -> Ok value
                    | _ -> Error "Invalid accepting configuration"
            | Some candidates -> Error(sprintf "Conflicting actions in s%d for %s: %A" state token.Kind candidates)
            | None -> Error(sprintf "Syntax error in s%d at token %s" state token.Lexeme)
        | _ -> Error "Incomplete parser configuration"
    step [0] [] (tokens @ [{ Kind = "$"; Lexeme = "end of input" }])

let parse table tokens =
    match parseBy table (fun _ -> ()) (fun _ _ -> ()) tokens with Ok () -> true | Error _ -> false

let test text =
    let accepted = parse ParseTable.table (tokenize text)
    printfn "%s: %b" text accepted
    accepted

## Testing

In [ ]:
checkNotebook (test "(1 + 2) * 3")

In [ ]:
checkNotebook (test "1 * 2 + 3 * (4 - 5) / 2")

In [ ]:
checkNotebook (test "11+22*(33-44)/(5-10*5/(4-3))")

In [ ]:
checkNotebook (not (test "1+2*3-"))

In [ ]:
checkNotebook (not (test "1+2*3-007"))